# LASSO und Ridge Regression zur Inflationsprognose
## Empirische Analyse mit deutschen Makrodaten

**Seminararbeit**: Aktuelle Fragen der Ökonometrie
**Betreuer**: Prof. Bernhard Schipp | Technische Universität Dresden

---

**Aufbau:**
1. Datenbeschaffung & -aufbereitung (ECB + Eurostat API)
2. Explorative Datenanalyse
3. Modelle: OLS, Ridge, LASSO
4. Cross-Validation mit TimeSeriesSplit (λ-Selektion)
5. Ergebnisvergleich (MSE, RMSE)
6. Visualisierungen: Koeffizientenpfade, Shrinkage-Plots

**Zielvariable (Y):** HVPI-Inflationsrate Deutschland (YoY, in %)
**Prädiktoren (X):** 33 makroökonomische Indikatoren aus 4 Kategorien → 155 Features mit Lags (nach NaN-Filter)


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import os, pathlib, sys
_ROOT = pathlib.Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data_preparation import get_raw_data, print_truncation_info
from src.data_preprocessing import build_feature_matrix, prepare_splits, transform_to_yoy
from src.training import fit_all_models
from src.evaluation import (
    compute_adaptive_oos, compute_compare_oos, compute_dm_tests,
    compute_horizon_analysis, compute_oos_predictions, compute_selection_stability,
    compute_single_split_inference,
    compute_stationarity_tests,
)
from src.reporting import (
    export_horizons_table, export_inference_table, export_results_table, export_sources_table,
    export_stationarity_table,
    fig_01_hvpi, fig_02_correlation, fig_02b_heatmap, fig_03_tscv,
    fig_04_forecast, fig_05_mse_comparison,
    fig_06_lasso_path, fig_07_ridge_path, fig_08_lasso_selection,
    fig_09_lasso_cv_path, fig_10_shrinkage,
    fig_11_rolling_rmse, fig_12_selection_stability, fig_13_horizons,
    print_summary, update_readmes,
)

config.setup_environment()
print("Alle Bibliotheken geladen. Projektwurzel:", _ROOT)

## 0. Datenfunktionen (ECB + Eurostat) 
Die Datenbeschaffungs- und Preprocessing-Funktionen liegen jetzt in `src/data_preparation.py` und `src/data_preprocessing.py`. Alle weiteren Pipeline-Stufen in `src/` (training, evaluation, reporting, pipeline).

In [ ]:
# Alle Datenfunktionen importiert aus src/data_preparation.py:
# get_raw_data(), load_all_data(), transform_to_yoy(), build_feature_matrix()
print("Datenfunktionen definiert (load_all_data, get_raw_data, transform_to_yoy, build_feature_matrix).")

## 1. Datenbeschaffung & -aufbereitung

In [ ]:
df_raw = get_raw_data(use_cache=True, verbose=True)
df_raw.head(3)

In [ ]:
df_yoy = transform_to_yoy(df_raw)
df_yoy.to_csv(config.DATA_PROCESSED)

X, y = build_feature_matrix(df_yoy, lags=config.LAGS, forecast_horizon=1,
                             test_months=config.TEST_MONTHS)
train_end = len(y) - config.TEST_MONTHS

print(f"Feature-Matrix:  {X.shape[0]} Beobachtungen x {X.shape[1]} Features")
print(f"Zielvariable:    {len(y)} Beobachtungen")
print(f"Zeitraum:        {y.index[0]:%Y-%m} - {y.index[-1]:%Y-%m}")
print(f"\nAnteil NaN in X: {X.isna().mean().mean():.1%}")

In [ ]:
print_truncation_info(df_raw, y)

### Stichproben-Erweiterung

**Lösung:** Eurostat lieferte IP_* und PPI_* mit `unit=I15` (Basisjahr 2015=100) nur bis **2023-12**. Das Basisjahr wurde auf **I21 (2021=100)** aktualisiert — inhaltlich identische Indexreihen, anderes Bezugsjahr.

**Neuer Datenstand nach Refresh:**

| Reihengruppe | Vorher | Nachher |
|---|---|---|
| Industrieproduktion (IP_*) | 2023-12 | **2026-04** |
| Produzentenpreise (PPI_*) | 2023-12 | **2026-04** |
| HVPI (Zielvariable) | 2025-12 | 2025-12 |
| LCI (Lohnkosten) | 2025-10 | 2025-10 |
| BS_Produktionserwart | 2024-09 | 2024-09 |

**Neues Sample-Limit:** `BS_Produktionserwart` endet bei 2024-09 → mit Lag 1 ist der letzte nutzbare Zielpunkt ca. **2024-10** (+9 Monate gegenüber 2024-01).

**Konsequenz:** Das Testfenster erstreckt sich nun auf die Disinflationsphase 2024, was die externe Validität des OOS-Vergleichs erhöht.

**Methodische Anmerkung:** Der Wechsel I15→I21 ändert nur den Bezugspunkt, nicht die Wachstumsraten; ist in der Arbeit als Datenquellen-Fußnote zu erwähnen.

## 2. Explorative Datenanalyse

In [ ]:
fig_01_hvpi(df_raw, df_yoy)

In [ ]:
fig_02_correlation(X, y)

### 2b. Multikollinearität der Prädiktoren

In [ ]:
fig_02b_heatmap(X, train_end)

### 2c. Stationaritätstests (ADF/KPSS)

YoY-Jahresveränderungsraten gelten als Standardtransformation zur Stationarisierung integrierter Makroreihen. Die folgenden ADF- und KPSS-Tests belegen dies testgestützt für HVPI und repräsentative Prädiktoren aus fünf Sektorgruppen.

**ADF** (H₀: Einheitswurzel): Verwerfung bei *p* < 0,05 → stationär.  
**KPSS** (H₀: Stationarität): Nicht-Verwerfung bei *p* ≥ 0,05 → stationär.

**Befund:** Industrieproduktion, PPI, Business Surveys und Arbeitslosigkeit sind nach YoY-Transformation klar stationär (ADF verwirft, KPSS verwirft nicht). HVPI-YoY zeigt *hohe Persistenz* (ADF p ≈ 0,015, KPSS p ≈ 0,044 — beide nahe am 5 %-Grenzwert), konsistent mit der bekannten Trägheit von Inflationsreihen (Stock & Watson 2007). Dies ist ein empirischer Befund, kein Verfahrensfehler: die YoY-Transformation reduziert die Persistenz des HVPI-Niveaus (ADF p > 0,99) deutlich und entspricht dem ökonometrischen Standard.

In [ ]:
stat_ctx = compute_stationarity_tests(df_raw, df_yoy)
export_stationarity_table(stat_ctx["df_stationarity"])

## 3. Modellschätzung: OLS, Ridge, LASSO

In [ ]:
splits  = prepare_splits(X, y, train_end)

y_train    = splits["y_train"];  y_test    = splits["y_test"]
X_train    = splits["X_train"];  X_train_s = splits["X_train_s"]

print(f"Trainingsdaten: {len(y_train)} Monate "
      f"({y_train.index[0].strftime('%Y-%m')} \u2013 {y_train.index[-1].strftime('%Y-%m')})")
print(f"Testdaten:      {len(y_test)} Monate "
      f"({y_test.index[0].strftime('%Y-%m')} \u2013 {y_test.index[-1].strftime('%Y-%m')})")
print(f"\nDimensionen: {X_train.shape[0]} Train \u00d7 {X_train.shape[1]} Features")
print(f"n < p: {'JA (hochdimensional)' if X_train.shape[0] < X_train.shape[1] else 'NEIN'}")

In [ ]:
print("Standardisierung abgeschlossen.")
print(f"Trainings-Mittelwerte ~0: {abs(X_train_s.mean(axis=0)).max():.2e}")
print(f"Trainings-Std ~1:         {abs(X_train_s.std(axis=0) - 1).max():.2e}")

In [ ]:
fig_03_tscv(splits["X_train_s"], config.TSCV)

## 3.5 Benchmarks: Random Walk & Lag-Modell (ADL)

Das **Lag-Modell (ADL)** verwendet ausschließlich HVPI-Eigen-Lags {1, 2, 3, 6, 12} als Prädiktoren — es ist damit ein *restringiertes AR-Modell* ohne exogene Regressoren. Die Bezeichnung „ADL" ist hier als Oberbegriff gewählt; da keine exogenen Größen eingehen, entspricht es inhaltlich einem autoregressiven Modell mit ausgewählten Lags.

In [ ]:
models_ctx = fit_all_models(X, y, splits, tscv=config.TSCV)

# Convenience-Aliase
ols           = models_ctx["ols"]
ridge_cv      = models_ctx["ridge_cv"]
lasso_cv      = models_ctx["lasso_cv"]
enet_cv       = models_ctx["enet_cv"]
alasso        = models_ctx["alasso"]
ar_model      = models_ctx["ar_model"]
lasso_plus_cv = models_ctx["lasso_plus_cv"]
lambda_ridge  = models_ctx["lambda_ridge"]
lambda_lasso  = models_ctx["lambda_lasso"]
lambda_enet   = models_ctx["lambda_enet"]
l1_ratio_enet = models_ctx["l1_ratio_enet"]
results       = models_ctx["results"]
selected      = models_ctx["selected"]
top_idx       = models_ctx["top_idx"]

## 4. Ergebnisvergleich

In [ ]:
print("=" * 75)
print("Ergebnisübersicht: Benchmarks vs. Regularisierungsmodelle")
print("=" * 75)
print(results.to_string())
print("=" * 75)

# Kombinierten Kontext aufbauen (wird von Reporting-Funktionen genutzt)
inf_ctx = compute_single_split_inference(models_ctx, splits)

ctx = {
    "df_raw": df_raw, "df_yoy": df_yoy,
    "X": X, "y": y, "train_end": train_end,
    **splits, **models_ctx, **inf_ctx, **stat_ctx,
}

### 4.1 Inferenz auf dem Einzelsplit (Block-Bootstrap + DM-Test)

**Forschungsfrage:** Ist die Rangfolge der Haupttabelle auf T=36 Testpunkten
statistisch belastbar?

Zwei komplementäre Methoden:
- **Block-Bootstrap 95%-KI** (zirkulär, $l=6\approx\sqrt{T}$, $B=2000$): quantifiziert die
  Stichprobenvariabilität des RMSE bei seriell korrelierten Fehlern.
- **Diebold-Mariano-Test** (HLN-korrigiert, $h=1$): testet, ob der quadratische
  Prognosefehler eines Modells signifikant kleiner ist als der des Random Walk.

**Erwartetes Ergebnis:** Bei $T=36$ ist die Testpower gering — Unterschiede in der
Größenordnung von RMSE/RW $\approx 1{,}4$ vs. $1{,}0$ sind i.d.R. nicht signifikant.
Die Block-Bootstrap-KIs geben an, in welcher Bandbreite der wahre RMSE mit 95%
Wahrscheinlichkeit liegt.

In [ ]:
export_inference_table(inf_ctx["df_inference"], splits["y_test"])

In [ ]:
fig_04_forecast(ctx)

In [ ]:
fig_05_mse_comparison(ctx)

## 4.5 Rolling-Origin Out-of-Sample

In [ ]:
oos_ctx = compute_oos_predictions(models_ctx, splits, X, y, train_end)
ctx.update(oos_ctx)

In [ ]:
fig_11_rolling_rmse(oos_ctx["oos_df"], oos_ctx["y_oos_ref"], oos_ctx["oos_rmse"])

In [ ]:
adap_ctx    = compute_adaptive_oos(X, y, splits, train_end, config.TSCV_INNER)
compare_ctx = compute_compare_oos(oos_ctx, adap_ctx, oos_ctx["y_oos_ref"])
ctx.update({**adap_ctx, **compare_ctx})

### 4.5.1 Einzelfenster vs. Rolling-Origin: Methodische Einordnung

Die Ergebnisse in Abschnitt 4 (fester Einzelsplit) und 4.5 (Rolling-Origin) weichen für
einige Modelle erheblich voneinander ab — das ist kein Widerspruch, sondern informativ:

| Modell | Einzelfenster RMSE | Rolling-Origin RMSE | Differenz |
|--------|-------------------:|--------------------:|----------:|
| Random Walk | 0.94 | 0.94 | ≈ 0 |
| AR (ADL) | 1.05 | 0.95 | −0.10 |
| LASSO+HVPI | 1.47 | 0.95 | −0.52 |
| LASSO | 1.83 | 1.09 | −0.74 |
| Elastic Net | 1.85 | 1.09 | −0.76 |
| Ridge | 1.96 | 1.16 | −0.80 |
| OLS | 3.40 | 2.34 | −1.06 |

**Mechanistische Erklärung der Diskrepanz:**

*Einzelfenster (fester Split 2021-06 bis 2024-10):* λ und alle Modellgewichte werden
**einmalig** auf dem Vor-PandemieDatensatz (bis 2021-05) geschätzt und dann starr auf
das Testfenster angewandt. Das Testfenster umfasst ausgerechnet den außergewöhnlichen
Energiepreis-/Inflationsschock 2021–2022 — ein Regime, das im Training nicht vorkam.
Modelle mit vielen Parametern (Ridge, LASSO mit Makro-Features) scheitern hier besonders,
weil die einmalig geschätzten Koeffizienten diesen Schock systematisch verpassen.

*Rolling-Origin (Expanding Window):* Das Modell wird **monatlich neu geschätzt** —
sobald die Schock-Monate im Trainingsfenster auftauchen, passen sich die Gewichte an.
AR und LASSO+HVPI profitieren am stärksten, weil die HVPI-Eigen-Lags den Inflations-
impuls schnell adaptieren.

**Konsequenz für die Ergebnisdarstellung:** Das Rolling-Origin-Design ist methodisch
robuster und wird als **Hauptergebnis** geführt. Der feste Einzelsplit dient als
Illustration der Regimeabhängigkeit: Er zeigt, wie anfällig reine Makro-Modelle ohne
Inflationspersistenz-Information für unerwartete Regimewechsel sind.

> *Beide Designs bestätigen den Kernbefund:* kein Makro-Modell ohne HVPI-Eigen-Lags
> schlägt den Random Walk; erst LASSO+HVPI und AR erreichen ihn (rolling) knapp.
> Die Signifikanz dieses Abstands prüft der Diebold-Mariano-Test im nächsten Abschnitt.


In [ ]:
dm_ctx = compute_dm_tests(oos_ctx, adap_ctx=adap_ctx)
ctx.update(dm_ctx)

### 4.6 Selektionsstabilität (LASSO)

In [ ]:
sel_ctx = compute_selection_stability(X, y, train_end, models_ctx["lambda_lasso"])
ctx.update(sel_ctx)

In [ ]:
fig_12_selection_stability(sel_ctx["sel_freq"], sel_ctx["n_windows"],
                           models_ctx["lambda_lasso"])

## 4.7 Mehrere Prognose-Horizonte

Vergleich der Modellgüte (RMSE) für h ∈ {1, 3, 6, 12} Monate voraus. Für jeden Horizont wird
`build_feature_matrix` mit `forecast_horizon=h` aufgerufen und λ via Cross-Validation neu bestimmt.
Die h-Schritt-Random-Walk-Prognose lautet ŷ_t = y_{t-h}.

In [ ]:
hor_ctx = compute_horizon_analysis(df_yoy, tscv=config.TSCV)
ctx.update(hor_ctx)

In [ ]:
fig_13_horizons(hor_ctx["df_horizons"])

**Befunde zur Horizont-Analyse:**  
Bei h = 6 sind LASSO und Elastic Net identisch (RMSE 3,876; 77 Selektion), weil `ElasticNetCV` für diesen Horizont `l1_ratio = 1,0` wählt und damit zur reinen L1-Strafe degeneriert.  
Die Selektionszahl steigt nicht-monoton (h=3: 15 → h=6: 77 → h=12: 6), weil das CV-optimale λ bei h=6 besonders klein ausfällt — der erhöhte Rauschanteil 6-Schritt-voraus lässt CV ein schwach regularisiertes Modell begünstigen, während bei h=12 das Signal so weit gedämpft ist, dass nur 6 Features das hohe λ überstehen.

## 5. Koeffizientenpfade & Variablenselektion

In [ ]:
feat_names = X.columns.tolist()
fig_06_lasso_path(splits["X_train_s"], splits["y_train"], lasso_cv, feat_names)
fig_07_ridge_path(splits["X_train_s"], splits["y_train"], ridge_cv, top_idx, feat_names)

In [ ]:
fig_08_lasso_selection(lasso_cv, X)

In [ ]:
fig_09_lasso_cv_path(lasso_cv)

In [ ]:
fig_10_shrinkage(ols, ridge_cv, lasso_cv, enet_cv)

## 6. Interpretation & Fazit

In [ ]:
print_summary(ctx)

### Interpretation der Ergebnisse

**1. Regularisierung behebt das OLS-Overfitting.** Bei p/n ≈ 0,69 (225 Beobachtungen, 155 Features) und ausgeprägter Multikollinearität (vgl. Konditionszahl in Abschnitt 2b) ist OLS unbrauchbar (Test-R² = −0,40). Ridge, LASSO und Elastic Net stabilisieren die Schätzung deutlich (Test-R² bis 0,77 (Adaptive LASSO) bzw. 0,74 (LASSO+HVPI), 0,59 ohne Eigen-Lags); LASSO erreicht dies mit nur 29 von 155 Variablen und liefert damit eine interpretierbare Selektion.

**2. Kein Makro-Modell schlägt den naiven Random Walk.** Der Random Walk (ŷ_t = y_{t−1}) hat mit RMSE ≈ 0,94 die beste Testgüte; LASSO liegt mit 1,83 rund 95 % darüber. Im robusteren Rolling-Origin-Design rücken die adaptiven Modelle (AR, LASSO+HVPI) bis an den RW heran (0,95 bzw. 0,95), schlagen ihn aber nicht klar — und über alle Horizonte h ∈ {1, 3, 6, 12} bleibt der RW die härteste Messlatte.

**3. Der Makro-Mehrwert über die Persistenz hinaus ist nahe null.** Erst das Modell mit den HVPI-Eigen-Lags (LASSO+HVPI) erreicht das RW-Niveau. Die reinen Makro-Modelle sind strukturell benachteiligt, weil ihnen der mit Abstand beste Einzelprädiktor — die letzte Inflationsrate — per Konstruktion fehlt.

**Einordnung.** Das Ergebnis ist konsistent mit der Literatur zur Inflationsprognose (Atkeson & Ohanian 2001; Stock & Watson 2007): strukturelle bzw. Phillips-Kurven-Modelle schlagen den naiven Benchmark in der Regel nicht. Die Kernaussage der Arbeit ist damit **nicht** „LASSO gewinnt", sondern: Regularisierung ist unverzichtbar, um in einem hochdimensionalen, kollinearen Prädiktorraum überhaupt stabil schätzen zu können — der ökonomische Prognosemehrwert gegenüber der reinen Persistenz bleibt jedoch gering.

**Limitationen:** Revidierte statt Echtzeit-Datenvintages; Testregime (2021–2024) maßgeblich vom Energiepreisschock geprägt — eingeschränkte externe Validität auf andere Inflationsregime. Prognoseunterschiede zum Random Walk sind bei T=36 statistisch nicht nachweisbar (Diebold-Mariano n.s.).

## 7. Aufbereitung & Export

Exportiert Tabellen als LaTeX-Fragmente (`results/*.tex`) und die Datenquellen-Tabelle als CSV + LaTeX für den Anhang. Abbildungen werden bei jedem `savefig`-Aufruf automatisch mit 300 DPI gespeichert (`savefig.dpi = 300` in den rcParams).

In [ ]:
export_results_table(models_ctx["results"], splits["y_test"])
export_inference_table(inf_ctx["df_inference"], splits["y_test"])
export_horizons_table(hor_ctx["df_horizons"])
export_sources_table()

## 8. README Auto-Synchronisation

Regeneriert den `Ergebnis-Überblick` in `README.md` aus den Live-Variablen
dieses Notebooks. Kein manuelles Pflegen von Zahlen nötig — einfach das
Notebook ausführen, dieser Block aktualisiert den README automatisch.


In [ ]:
update_readmes(ctx)